In [2]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

# convert section to lowercase because they are inconsistent across both datasets
events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)


In [9]:
print("\nColumns:")
print(events.column_names)
print("\nFeatures:")
print(events.features)

import random
from pprint import pprint

n_samples = 1

indices = random.sample(range(len(events)), n_samples)

print(f"\n{n_samples} Random events:")
for idx in indices:
    sample = events[idx]
    print(f"\nSample[{idx}]:")
    pprint(sample)

# count articles in each section
from collections import Counter

section_counts = Counter(events["section"])

print("\nSection counts:")
for section, count in section_counts.most_common():
    print(f"\t{section}: {count}")



Columns:
['month', 'year', 'day', 'section', 'content', 'instruction', 'response']

Features:
{'month': Value('string'), 'year': Value('int64'), 'day': Value('int64'), 'section': Value('string'), 'content': Value('string'), 'instruction': Value('string'), 'response': Value('string')}

1 Random events:

Sample[10695]:
{'content': 'Deportation in the second Trump administration\n'
            'U.S. federal judge Angel Kelley of Massachusetts blocks the Trump '
            "administration's plan to end temporary protected status for South "
            'Sudanese nationals, issuing an administrative stay that prevents '
            'the expiration of deportation protections while a legal challenge '
            'proceeds. (Reuters)',
 'day': 30,
 'instruction': 'On December 30, 2025, U.S. federal judge Angel Kelley issued '
                'an administrative stay that blocked the Trump '
                "administration's plan to end Temporary Protected Status for "
                'South 

In [10]:
import requests, time

ollama_base_url = "http://localhost:11434/api/"
embedding_models = [
    "mxbai-embed-large",
    "nomic-embed-text"
]

for embedding_model in embedding_models:

    start = time.time()
    print(f"\n**** Embeddings using model {embedding_model}")
    response = requests.post(
        ollama_base_url + "embeddings",
        json={
            "model": embedding_model,
            "prompt": sample["instruction"]
        }
    )
    end = time.time()
    print(f"{len(response.json()["embedding"])} dimensions generated in {round(end-start, 3)}s")


**** Embeddings using model mxbai-embed-large
1024 dimensions generated in 0.941s

**** Embeddings using model nomic-embed-text
768 dimensions generated in 0.598s


In [11]:
llm_models = [
    "qwen3.5:9b",
    "llama3.2:3b",
    "llama3.2:1b",
    "gpt-oss:20b"
]


for llm_model in llm_models:
    start = time.time()

    print(f"\n\n*** Trial generate with {llm_model}:")
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Summarize the following text clearly and concisely:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Summary: " + response.json()["response"])
    end = time.time()
    print(f"Summary Time: {round(end - start)}s") 

    start = time.time()
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Give three main concise and clear points from the text:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Main points:\n" + response.json()["response"])

    end = time.time()
    print(f"Main points time: {round(end - start)}s") 



*** Trial generate with qwen3.5:9b:
Summary: Federal Judge Angel Kelley has blocked the second Trump administration's plan to end Temporary Protected Status for South Sudanese nationals, preserving their deportation protections pending a legal challenge.
Summary Time: 25s
Main points:
1. A U.S. federal judge blocked the administration's plan to end temporary protected status for South Sudanese nationals.
2. An administrative stay was issued to prevent the expiration of these deportation protections.
3. The protections remain in place while the legal challenge proceeds.
Main points time: 27s


*** Trial generate with llama3.2:3b:
Summary: A US federal judge blocked the Trump administration's plan to end temporary protected status for South Sudanese nationals, granting them deportation protection until a court case can be resolved.
Summary Time: 2s
Main points:
Here are three concise and clear points from the text:

1. A US federal judge blocked the Trump administration's plan to end t